In [3]:
import sys
print(sys.executable)


d:\Dev\AMVerge\backend\venv\Scripts\python.exe


In [8]:
import site
print(site.getsitepackages())

['d:\\Dev\\AMVerge\\backend\\venv', 'd:\\Dev\\AMVerge\\backend\\venv\\Lib\\site-packages']


In [9]:
import sys
print(sys.path)

['C:\\Python313\\python313.zip', 'C:\\Python313\\DLLs', 'C:\\Python313\\Lib', 'C:\\Python313', 'd:\\Dev\\AMVerge\\backend\\venv', '', 'd:\\Dev\\AMVerge\\backend\\venv\\Lib\\site-packages']


In [14]:
import os
import torch

os.add_dll_directory(
    r"C:\ffmpeg-shared\ffmpeg-8.1.1-full_build-shared\bin"
)

import nelux

In [15]:
import torch
import nelux
print(nelux.__file__)

d:\Dev\AMVerge\backend\venv\Lib\site-packages\nelux\__init__.py


In [20]:
import os
import torch
try:
    from nelux import VideoReader
except ImportError as exc:
    raise ImportError(
        "nelux is not installed. Install it before using nelux decode path."
    ) from exc
import numpy as np
from tqdm import tqdm
import sys
from pathlib import Path

FRAME_WIDTH = 48
FRAME_HEIGHT = 27
FRAME_CHANNELS = 3
FRAME_BYTES = FRAME_WIDTH * FRAME_HEIGHT * FRAME_CHANNELS
WINDOW_SIZE = 100               # how many frames the AI should process at once (batch size)
OVERLAP = 50                    # The overlap that the AI should have for context for each batch
STRIDE = WINDOW_SIZE - OVERLAP  # how much 

def check_if_path_exists(path_str):
    if not os.path.exists(path_str):
        raise FileNotFoundError(f"Path does not exist: {path_str}")
    return True

def decode_video_frames_nelux(input_video):
    """Decode frames with nelux into the same shape consumed by TransNetV2.

    Returns a numpy array with shape:
    (num_frames, FRAME_HEIGHT, FRAME_WIDTH, FRAME_CHANNELS), dtype=uint8
    """
    check_if_path_exists(input_video)

    decode_accelerator = "nvdec" if torch.cuda.is_available() else None
    reader = VideoReader(
        str(input_video),
        decode_accelerator=decode_accelerator,
        resize=(FRAME_WIDTH, FRAME_HEIGHT),
    )

    total_frames = len(reader)
    frames = np.empty(
        (total_frames, FRAME_HEIGHT, FRAME_WIDTH, FRAME_CHANNELS),
        dtype=np.uint8,
    )

    actual_frames = 0
    with tqdm(
        desc="Decoding video with nelux..",
        unit="frames",
        file=sys.stdout,
        total=total_frames,
    ) as pbar:
        for i in range(total_frames):
            frame = reader.read_frame()
            if frame is None:
                break

            if isinstance(frame, torch.Tensor):
                frame_np = frame.detach().to("cpu").numpy().astype(np.uint8, copy=False)
            else:
                frame_np = np.asarray(frame, dtype=np.uint8)

            # Normalize to HWC layout expected by TransNetV2 input preprocessing.
            if frame_np.ndim != 3:
                raise ValueError(f"Unexpected frame rank from nelux: {frame_np.ndim}")

            if frame_np.shape[0] == FRAME_CHANNELS and frame_np.shape[-1] != FRAME_CHANNELS:
                frame_np = np.transpose(frame_np, (1, 2, 0))

            if frame_np.shape != (FRAME_HEIGHT, FRAME_WIDTH, FRAME_CHANNELS):
                raise ValueError(
                    "Unexpected frame shape from nelux. "
                    f"Got {frame_np.shape}, expected "
                    f"({FRAME_HEIGHT}, {FRAME_WIDTH}, {FRAME_CHANNELS})."
                )

            frames[i] = frame_np
            actual_frames += 1
            pbar.update(1)

    if actual_frames < total_frames:
        frames = frames[:actual_frames]

    return frames

BASE_DIR = Path.cwd().resolve()
print(f"BASE_DIR: {BASE_DIR}")

INPUT_DIR = BASE_DIR / "input"
INPUT_FILE = INPUT_DIR / "franxx_ep.mkv"

frames = decode_video_frames_nelux(INPUT_FILE)

print(f"len(frames): {len(frames)}")
print(frames.shape)
print(frames.dtype)

print(frames.min())
print(frames.max())
print(frames.mean())

BASE_DIR: D:\Dev\AMVerge\backend\notebooks\ffmpeg_practice
Decoding video with nelux..: 100%|██████████| 34574/34574 [00:15<00:00, 2272.19frames/s]
len(frames): 34574
(34574, 27, 48, 3)
uint8
0
255
57.544696273526505
